# Khipus.ai
## Introduction to Generative AI and Large Language Models
### Assignment 3:  Create an Index in pinecone
### Name: (add your name)
<span>© Copyright Notice 2025, Khipus.ai - All Rights Reserved.</span>

In [ ]:
# Ejecutar en la terminal del entorno de la unidad 3:
# python -m pip install -r requirements.txt

In [ ]:
from time import monotonic, sleep
from pinecone import Pinecone, ServerlessSpec

PINECONE_API_KEY = '********-****-****-****-************'
PINECONE_INDEX_NAME = "khipus-unit3-3072"
EMBEDDING_DIMENSIONS = 3072  # text-embedding-3-large del proyecto Foundry

if not PINECONE_API_KEY.strip() or PINECONE_API_KEY.startswith("YOUR_"):
    raise ValueError("Completa PINECONE_API_KEY antes de continuar.")
pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
# Crear solo si falta; nunca borrar o recrear un índice existente.
if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

deadline = monotonic() + 60
while True:
    description = pc.describe_index(PINECONE_INDEX_NAME)
    if description.dimension != EMBEDDING_DIMENSIONS or description.metric != "cosine":
        raise ValueError(
            f"El índice {PINECONE_INDEX_NAME!r} no coincide con "
            f"{EMBEDDING_DIMENSIONS} dimensiones y métrica cosine. "
            "Elige otro PINECONE_INDEX_NAME; no borres un índice con datos."
        )
    if description.status["ready"]:
        break
    if monotonic() >= deadline:
        raise TimeoutError("Pinecone aún está preparando el índice. Espera y vuelve a ejecutar esta celda.")
    sleep(2)

index = pc.Index(host=description.host)
print("Índice preparado:", PINECONE_INDEX_NAME)